# 07 — Topic modeling sul Guardian (testo completo)

Il BERTopic sul NYT (`04_SemanticAnalysis.ipynb` §5) lavora su titolo + abstract + lead (~50 parole ad articolo), perché il testo integrale non è disponibile. Per il Guardian abbiamo il `bodyText` completo: qui verifichiamo se **il testo completo produce topic più netti** del testo breve.

Per rispondere serve un confronto a parità di tutto il resto, quindi sullo **stesso corpus Guardian** addestriamo due modelli con gli stessi parametri del NYT:

- **breve** — titolo + sottotitolo + prime `LEAD_WORDS` parole: replica la condizione del NYT;
- **completo** — tutto l'articolo.

Input: `articles_ice_relevant.parquet`, solo gli articoli con `is_ice_relevant` (filtro di rilevanza di `05_guardian.ipynb` §5, modello B): fare topic modeling sulle menzioni di passaggio troverebbe i temi di *altri* articoli, non quelli di ICE.

**Attenzione al troncamento.** `all-MiniLM-L6-v2` legge al massimo 256 token (~190 parole) e ignora il resto: passargli un articolo di 1.000 parole equivale a passargli l'attacco, cioè di nuovo la condizione "breve". Per usare davvero il testo completo l'articolo va spezzato in **blocchi** (§2), ogni blocco va trasformato in embedding e l'articolo è rappresentato dalla **media dei suoi blocchi** (§3).

## 0. Setup

In [ ]:
import logging
import re
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_mutual_info_score
from umap import UMAP

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "guardian"
PLOTS_DIR = PROCESSED_DIR / "plots"
IN_RELEVANT = PROCESSED_DIR / "articles_ice_relevant.parquet"
CHUNK_EMBED_CACHE = PROCESSED_DIR / "topic_chunk_embeddings_minilm.parquet"
OUT_TOPICS = PROCESSED_DIR / "ice_topics.parquet"
OUT_TOPIC_INFO = PROCESSED_DIR / "ice_topic_info.csv"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
PERIOD_ORDER = ["2003-2016", "2017-2020", "2021-2024", "2025-presente"]

sns.set_theme(style="whitegrid", palette="muted")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("guardian_topics")
for noisy in ("httpx", "sentence_transformers", "huggingface_hub", "BERTopic"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 140)
print(f"BASE_DIR: {BASE_DIR}")

## 1. Corpus

Solo articoli `is_ice_relevant`. I **live-blog** ("as it happened") sono un caso a parte: sono lunghissimi e coprono decine di notizie diverse nella stessa pagina, quindi non hanno *un* tema. Restano nel corpus, ma li marchiamo (`is_live`) per vedere dove finiscono.

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")

df = pd.read_parquet(IN_RELEVANT)
df = df[df["is_ice_relevant"]].reset_index(drop=True)
df["standfirst_clean"] = df["standfirst"].fillna("").str.replace(TAG_RE, " ", regex=True).str.strip()
df["is_live"] = df["id"].str.contains("/live/", regex=False)
df["n_words"] = df["body_text"].fillna("").str.split().str.len()
log.info("Articoli rilevanti: %d (live-blog: %d)", len(df), df["is_live"].sum())

display(df.groupby("period").agg(articoli=("id", "size"), live=("is_live", "sum"), parole_mediana=("n_words", "median")).reindex(PERIOD_ORDER))
df["n_words"].describe(percentiles=[0.1, 0.5, 0.9]).round(0).to_frame("parole per articolo").T

## 2. Le due versioni del testo e i blocchi

- `text_short`: titolo + sottotitolo + prime `LEAD_WORDS` parole (≈ abstract + lead del NYT).
- `text_full`: titolo + sottotitolo + tutto il corpo.

Il `body_text` del Guardian non ha a capo, quindi i blocchi si costruiscono accumulando **frasi intere** fino a `CHUNK_WORDS` parole (sotto il limite di 256 token del modello, così nessun blocco viene troncato).

In [ ]:
LEAD_WORDS = 60
CHUNK_WORDS = 150
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])[\"”’']?\s+(?=[A-Z“‘\"'])")

head = (df["headline"].fillna("") + ". " + df["standfirst_clean"]).str.strip()
body = df["body_text"].fillna("")
df["text_short"] = head + " " + body.map(lambda b: " ".join(b.split()[:LEAD_WORDS]))
df["text_full"] = head + " " + body


def chunk_text(text: str, max_words: int = CHUNK_WORDS) -> list[str]:
    chunks, current, n = [], [], 0
    for sent in SENT_SPLIT_RE.split(text):
        w = len(sent.split())
        if current and n + w > max_words:
            chunks.append(" ".join(current))
            current, n = [], 0
        current.append(sent)
        n += w
    if current:
        chunks.append(" ".join(current))
    return chunks


chunks = pd.DataFrame(
    [(i, j, c) for i, text in enumerate(df["text_full"]) for j, c in enumerate(chunk_text(text))],
    columns=["doc_idx", "chunk_idx", "text"],
)
per_doc = chunks.groupby("doc_idx").size()
log.info("Blocchi: %d (mediana %d per articolo, max %d)", len(chunks), per_doc.median(), per_doc.max())
print("parole per blocco:", chunks["text"].str.split().str.len().describe(percentiles=[0.5, 0.95]).round(0).to_dict())
print(f"parole mediane: breve {df['text_short'].str.split().str.len().median():.0f} · completo {df['text_full'].str.split().str.len().median():.0f}")

## 3. Embedding

- **breve**: un embedding per articolo, su `text_short`.
- **completo**: un embedding per blocco, poi **media dei blocchi** dell'articolo (rinormalizzata). Gli embedding dei blocchi sono in cache su disco: si ricalcolano solo se cambia l'insieme dei blocchi.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)


def embed_chunks() -> np.ndarray:
    key = pd.util.hash_pandas_object(chunks["text"], index=False).sum()
    if CHUNK_EMBED_CACHE.exists():
        cached = pd.read_parquet(CHUNK_EMBED_CACHE)
        if len(cached) == len(chunks) and cached.attrs.get("key") == str(key):
            return cached.to_numpy()
    log.info("Embedding di %d blocchi con %s...", len(chunks), EMBEDDING_MODEL)
    vecs = embedding_model.encode(chunks["text"].tolist(), batch_size=128, show_progress_bar=False, normalize_embeddings=True)
    out = pd.DataFrame(vecs, columns=[f"e{i}" for i in range(vecs.shape[1])])
    out.attrs["key"] = str(key)
    out.to_parquet(CHUNK_EMBED_CACHE)
    return vecs


chunk_emb = embed_chunks()
emb_full = pd.DataFrame(chunk_emb).groupby(chunks["doc_idx"].values).mean().to_numpy()
emb_full = emb_full / np.linalg.norm(emb_full, axis=1, keepdims=True)
emb_short = embedding_model.encode(df["text_short"].tolist(), batch_size=128, show_progress_bar=False, normalize_embeddings=True)
print(f"breve: {emb_short.shape} · completo: {emb_full.shape} (da {len(chunk_emb)} blocchi)")

## 4. BERTopic — stessi parametri del NYT, due versioni del testo

Parametri identici a `04_SemanticAnalysis.ipynb` §5 (`min_topic_size=15`, `UMAP(random_state=42)`, bigrammi, `reduce_outliers` a soglia 0,4), così l'unica differenza tra i due modelli è il testo. Stopword: quelle del NYT più alcune parole di servizio del Guardian (`told`, `guardian`…). `ice` resta escluso dalle etichette perché compare ovunque.

Le parole rappresentative (c-TF-IDF) sono calcolate sul testo della rispettiva versione: sul testo completo il vocabolario a disposizione è ~10 volte più ampio.

**Più seed, non uno solo.** UMAP è stocastico: con un seed diverso la mappa a 5 dimensioni cambia e HDBSCAN può trovare gruppi diversi. Un confronto su un solo seed può dipendere dal caso (con `seed=42` il modello breve collassa in un unico topic gigante, con altri seed no), quindi ogni versione viene addestrata su tutti i `SEEDS`. Il modello finale è quello sul testo completo con `REFERENCE_SEED = 42`, lo stesso seed del NYT.

In [ ]:
CUSTOM_STOPWORDS = list(ENGLISH_STOP_WORDS | {"ice", "new", "said", "says", "mr", "ms", "year", "years", "old",
                                              "told", "guardian", "people", "just", "like", "according", "including"})
MIN_TOPIC_SIZE = 15
OUTLIER_THRESHOLD = 0.4
SEEDS = [0, 1, 7, 42, 123]
REFERENCE_SEED = 42


def fit_topics(docs: list[str], embeddings: np.ndarray, seed: int) -> tuple[BERTopic, list[int], np.ndarray]:
    vectorizer = CountVectorizer(stop_words=CUSTOM_STOPWORDS, ngram_range=(1, 2), min_df=3)
    model = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        vectorizer_model=vectorizer,
        min_topic_size=MIN_TOPIC_SIZE,
    )
    raw, _ = model.fit_transform(docs, embeddings)
    was_outlier = np.array(raw) == -1
    if was_outlier.any():
        topics = model.reduce_outliers(docs, raw, strategy="embeddings", embeddings=embeddings, threshold=OUTLIER_THRESHOLD)
        model.update_topics(docs, topics=topics, vectorizer_model=vectorizer)
    else:
        topics = raw
    return model, list(topics), was_outlier


runs = {}
for seed in SEEDS:
    for name, col, emb in [("breve", "text_short", emb_short), ("completo", "text_full", emb_full)]:
        model, topics, was_outlier = fit_topics(df[col].tolist(), emb, seed)
        runs[(name, seed)] = {"model": model, "topics": np.array(topics), "emb": emb, "was_outlier": was_outlier}
        log.info("seed %3d · %-8s: %2d topic, outlier %.1f%% -> %.1f%%", seed, name, len(set(topics)) - (-1 in topics),
                 100 * was_outlier.mean(), 100 * np.mean(np.array(topics) == -1))

## 5. Confronto breve vs completo

Misure, calcolate allo stesso modo per ogni modello (versione × seed):

- **n. topic** e **outlier** (prima e dopo la riassegnazione): quanto del corpus il modello riesce a organizzare.
- **Coerenza (NPMI)**: per le 10 parole più rappresentative di ogni topic, quanto spesso compaiono *insieme* rispetto al caso. Riferimento comune = i **blocchi** da ~150 parole del corpus (una finestra di co-occorrenza uguale per entrambi i modelli, indipendente dal testo su cui sono stati addestrati). Da −1 a 1; più alto = parole che stanno davvero insieme.
- **Diversità**: quota di parole uniche tra le top-10 di tutti i topic. Bassa = topic che ripetono le stesse parole.
- **Silhouette** (coseno) nello spazio degli embedding *completi*, per entrambi: quanto i gruppi sono separati rispetto al contenuto dell'intero articolo.
- **Stabilità**: AMI medio tra le partizioni ottenute con seed diversi, *dentro* la stessa versione (1 = stessi gruppi qualunque sia il seed, 0 = gruppi indipendenti). Un modello i cui topic cambiano a ogni esecuzione non è interpretabile, per quanto coerenti siano le parole.
- **Collassi**: numero di seed in cui un solo topic assorbe più di metà del corpus.

In [ ]:
from sklearn.metrics import silhouette_score

TOP_WORDS = 10
ref_vectorizer = CountVectorizer(stop_words=CUSTOM_STOPWORDS, ngram_range=(1, 2), min_df=3, binary=True)
ref_matrix = ref_vectorizer.fit_transform(chunks["text"]).tocsc()
ref_vocab = ref_vectorizer.vocabulary_
n_ref = ref_matrix.shape[0]


def topic_words(model: BERTopic, topic: int, n: int = TOP_WORDS) -> list[str]:
    return [w for w, _ in model.get_topic(topic)[:n] if w]


def npmi(words: list[str]) -> float:
    cols = [ref_vocab[w] for w in words if w in ref_vocab]
    scores = []
    for a, b in combinations(cols, 2):
        pa, pb = ref_matrix[:, a].sum() / n_ref, ref_matrix[:, b].sum() / n_ref
        pab = ref_matrix[:, a].multiply(ref_matrix[:, b]).sum() / n_ref
        scores.append(-1.0 if pab == 0 else np.log(pab / (pa * pb)) / -np.log(pab))
    return float(np.mean(scores)) if scores else np.nan


def summarize(name: str, seed: int) -> dict:
    run = runs[(name, seed)]
    model, topics = run["model"], run["topics"]
    ids = sorted(t for t in set(topics) if t != -1)
    words = [topic_words(model, t) for t in ids]
    sizes = pd.Series(topics[topics != -1]).value_counts()
    assigned = topics != -1
    return {
        "versione": name, "seed": seed,
        "topic": len(ids),
        "outlier prima %": 100 * run["was_outlier"].mean(),
        "outlier dopo %": 100 * (~assigned).mean(),
        "topic più grande %": 100 * sizes.max() / len(topics),
        "dimensione mediana": sizes.median(),
        "coerenza NPMI": np.nanmean([npmi(w) for w in words]),
        "diversità": len({w for ws in words for w in ws}) / sum(len(ws) for ws in words),
        "silhouette (emb. completi)": silhouette_score(emb_full[assigned], topics[assigned], metric="cosine"),
    }


comparison = pd.DataFrame([summarize(name, seed) for name in ("breve", "completo") for seed in SEEDS]).set_index(["versione", "seed"])
display(comparison.round(3))

metric_cols = ["topic", "outlier prima %", "topic più grande %", "coerenza NPMI", "diversità", "silhouette (emb. completi)"]
summary = comparison.groupby("versione")[metric_cols].agg(["mean", "min", "max"]).round(3)
stability = {
    name: np.mean([adjusted_mutual_info_score(runs[(name, a)]["topics"], runs[(name, b)]["topics"]) for a, b in combinations(SEEDS, 2)])
    for name in ("breve", "completo")
}
collapses = (comparison["topic più grande %"] > 50).groupby("versione").sum()
between = np.mean([adjusted_mutual_info_score(runs[("breve", s)]["topics"], runs[("completo", s)]["topics"]) for s in SEEDS])
display(summary.T)
print(f"Stabilità (AMI medio tra seed): breve {stability['breve']:.3f} · completo {stability['completo']:.3f}")
print(f"Collassi (un topic > 50% del corpus): breve {collapses['breve']}/{len(SEEDS)} · completo {collapses['completo']}/{len(SEEDS)}")
print(f"AMI medio tra breve e completo a parità di seed: {between:.3f}")

## 6. I topic del modello sul testo completo

Modello di riferimento: testo completo, `REFERENCE_SEED`. Per ogni topic: numero di articoli, quota di live-blog, parole più rappresentative e i titoli dei 3 articoli più vicini al centro del topic (più leggibili del testo integrale restituito da BERTopic).

In [ ]:
def describe_topics(name: str, seed: int) -> pd.DataFrame:
    run = runs[(name, seed)]
    model, topics, emb = run["model"], run["topics"], run["emb"]
    rows = []
    for t in sorted(set(topics) - {-1}, key=lambda t: -(topics == t).sum()):
        idx = np.where(topics == t)[0]
        centroid = emb[idx].mean(axis=0)
        closest = idx[np.argsort(-(emb[idx] @ centroid))[:3]]
        rows.append({
            "topic": t, "articoli": len(idx), "% live": round(100 * df["is_live"].values[idx].mean()),
            "parole": ", ".join(topic_words(model, t, 8)),
            "titoli più centrali": " | ".join(df["headline"].values[closest]),
        })
    return pd.DataFrame(rows).set_index("topic")


final = runs[("completo", REFERENCE_SEED)]
topic_table = describe_topics("completo", REFERENCE_SEED)
df["topic"] = final["topics"]
# Articoli che HDBSCAN aveva lasciato fuori e che reduce_outliers ha assegnato al topic più simile: appartenenza più debole
df["topic_was_outlier"] = final["was_outlier"]
labels = {t: f"{t}: " + ", ".join(topic_words(final["model"], t, 3)) for t in topic_table.index}
labels[-1] = "-1: outlier"
df["topic_label"] = df["topic"].map(labels)
display(topic_table)

Per confronto, i topic del modello sul testo breve (stesso corpus). Si mostra il seed con più topic, cioè il caso *più favorevole* al testo breve tra quelli provati — non quello di riferimento, che per il breve è un collasso:

In [ ]:
best_short_seed = comparison.loc["breve", "topic"].idxmax()
print(f"seed {best_short_seed}")
display(describe_topics("breve", best_short_seed))

### 6.1 Peso dei topic per periodo

I periodi hanno dimensioni molto diverse (il 2025-presente da solo è ~70% del corpus), quindi ogni colonna è normalizzata: **quota degli articoli di quel periodo** in ciascun topic.

In [ ]:
topic_period = pd.crosstab(df["topic_label"], df["period"], normalize="columns").reindex(columns=PERIOD_ORDER) * 100
topic_period = topic_period.drop(index=labels[-1], errors="ignore")
topic_period = topic_period.loc[topic_period.max(axis=1).sort_values(ascending=False).index]
period_sizes = df["period"].value_counts().reindex(PERIOD_ORDER)

fig, ax = plt.subplots(figsize=(10, 0.42 * len(topic_period) + 2))
sns.heatmap(topic_period, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% articoli del periodo"}, ax=ax)
ax.set_xticklabels([f"{p}\n(n={period_sizes[p]})" for p in PERIOD_ORDER], rotation=0)
ax.set_title("Guardian (testo completo) — peso di ciascun topic per periodo (%)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "topics_by_period.png", dpi=150)
plt.show()

## 7. Salvataggio

`ice_topics.parquet`: un articolo per riga con il topic del modello di riferimento (testo completo, `REFERENCE_SEED`) e `topic_was_outlier` (vero se l'articolo era outlier prima della riassegnazione). `ice_topic_info.csv`: tabella dei topic.

In [ ]:
out = df[["id", "pub_date", "year", "period", "is_live", "topic", "topic_label", "topic_was_outlier"]].copy()
out.to_parquet(OUT_TOPICS, index=False)
topic_table.reset_index().to_csv(OUT_TOPIC_INFO, index=False)
log.info("Scritti %s (%d articoli, %d topic) e %s", OUT_TOPICS.relative_to(BASE_DIR), len(out), len(topic_table), OUT_TOPIC_INFO.relative_to(BASE_DIR))

## 8. Lettura dei risultati (run 2026-09-30)

**Il testo completo è più *affidabile*, non più *coerente*.**

- **Stabilità** — è la differenza netta. Con il testo completo i topic sono quasi gli stessi a ogni seed (25-27 topic, AMI tra seed 0,85, nessun collasso). Con il testo breve cambiano molto (AMI 0,39) e in 2 seed su 5 il modello collassa: un solo topic con il 97% degli articoli.
- **Coerenza delle parole** — nessun vantaggio. Quando il modello breve non collassa, le sue parole sono leggermente *più* coerenti (NPMI 0,24-0,25 contro 0,21) e più varie tra topic (diversità 0,75-0,77 contro 0,64-0,66). Il testo completo usa un vocabolario generico (`immigration`, `trump`, `detention`) che ricorre in molti topic.
- **Separazione dei gruppi** — silhouette bassa in entrambi i casi (0,08 contro 0,03): gli articoli sono tutti su ICE e i confini tra topic sono sfumati. Misura calcolata sugli embedding completi, quindi strutturalmente favorevole al modello completo.
- **Cosa cambia nei contenuti** — il breve, quando funziona, trova soprattutto **eventi** (raid a Los Angeles, Minneapolis, Khalil, Ábrego García). Il completo trova gli stessi eventi più dei **temi trasversali** che dal solo attacco non emergono: condizioni di detenzione, morti in custodia, lavoratori e raid, sorveglianza e dati (Palantir), tribunali dell'immigrazione, città santuario.

**Limiti.**

- Il 39% degli articoli era outlier ed è stato assegnato al topic più simile (`topic_was_outlier`). Con embedding mediati sui blocchi le similarità sono tutte alte (coseno medio tra articoli 0,49 contro 0,30 del breve), quindi la soglia 0,4 non filtra quasi nulla: in alcuni topic (16, 9, 23) oltre metà degli articoli è riassegnata. Per analisi che richiedono appartenenze solide, usare solo `topic_was_outlier == False`.
- Il topic 6 (`trump, president, house`) è per il 28% live-blog: è un contenitore di cronaca politica generale, non un tema su ICE.
- Periodi pre-2025 piccoli (108 / 283 / 90 articoli): le quote per periodo nella heatmap hanno molta incertezza e i topic-evento del 2025+ valgono 0 per costruzione.
- Questo confronto dice cosa succede **sul Guardian**. Non dimostra che i topic del NYT (notebook 04, testo breve) siano sbagliati, ma suggerisce di verificarne la stabilità su più seed prima di interpretarli.
